# Session 3 - Value-at-Risk on a Single Stock: Apple (AAPL)
**Financial Risk Management and Engineering with Python**
Aarhus BSS Summer University 2026

---
We estimate 1-day Value-at-Risk for a **\$1,000,000 long position in AAPL** using the
**last 500 trading days** of real market data, applying all three canonical methods and
then comparing them:

1. **Parametric (variance-covariance)** - assume normal returns.
2. **Historical simulation** - use the empirical distribution, no assumption.
3. **Monte Carlo** - simulate returns from a fitted model.

$$\text{VaR}_{\alpha,T} = \text{loss exceeded with probability } (1-\alpha) \text{ over horizon } T$$


In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import norm
import yfinance as yf

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': 0.3})

# --- Single-stock study: Apple (AAPL), last 500 trading days ------------------
TICKER   = 'AAPL'
POSITION = 1_000_000      # $1,000,000 long position in AAPL
WINDOW   = 500            # last 500 trading days
ALPHA    = 0.99           # primary confidence level

# Pull extra history so the 500-day window is complete and a rolling backtest has room
raw = yf.download(TICKER, start='2019-01-01', end='2025-06-30',
                  auto_adjust=True, progress=False)['Close']
if isinstance(raw, pd.DataFrame):      # newer yfinance returns a 1-column frame
    raw = raw.iloc[:, 0]
raw = raw.dropna()

ret_all = np.log(raw / raw.shift(1)).dropna()    # daily log returns, full history
ret     = ret_all.iloc[-WINDOW:]                 # THE study sample: last 500 trading days
pnl     = ret * POSITION                         # daily P&L in $ on the position

print(f"Stock: {TICKER}    Position: ${POSITION:,.0f}")
print(f"Study window: {ret.index[0].date()} to {ret.index[-1].date()}  ({len(ret)} trading days)")
print(f"Daily mean:  {ret.mean()*100:.4f}%    Daily vol: {ret.std()*100:.4f}%")
print(f"Annualised vol: {ret.std()*np.sqrt(252)*100:.1f}%")
print(f"Skew: {stats.skew(ret):.2f}    Excess kurtosis: {stats.kurtosis(ret):.2f}  (0 = normal)")



1 Failed download:
['AAPL']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


Stock: AAPL    Position: $1,000,000


IndexError: index 0 is out of bounds for axis 0 with size 0

## 1. Parametric (Variance-Covariance) VaR
Fit a normal to the 500-day returns and read the quantile:
$$\text{VaR}_\alpha = -(\mu + z_{1-\alpha}\,\sigma)\,V, \qquad z_\alpha = \Phi^{-1}(\alpha)$$


In [ ]:
mu  = ret.mean()
sig = ret.std()

def parametric_var(alpha):
    z = norm.ppf(alpha)               # 2.326 at 99%
    return -(mu - z*sig) * POSITION   # positive number = loss

for a in [0.95, 0.99]:
    v = parametric_var(a)
    print(f"{a*100:.0f}% 1-day Parametric VaR: ${v:>9,.0f}  ({v/POSITION*100:.2f}%)")

var_param = parametric_var(ALPHA)
print(f"\nAssumes AAPL daily returns ~ Normal(mu={mu*100:.3f}%, sigma={sig*100:.2f}%).")
print("Watch the kurtosis above: if tails are fat, this understates the true 99% loss.")


## 2. Historical Simulation VaR
No distribution assumed - just the empirical quantile of the last 500 daily P&Ls:
$$\text{VaR}^{HS}_\alpha = -\,Q_{1-\alpha}\big(\text{last 500 daily P\&L}\big)$$


In [ ]:
def historical_var(alpha):
    return -np.percentile(pnl, (1-alpha)*100)

for a in [0.95, 0.99]:
    v = historical_var(a)
    print(f"{a*100:.0f}% 1-day Historical VaR: ${v:>9,.0f}  ({v/POSITION*100:.2f}%)")

var_hist = historical_var(ALPHA)
print("\nWorst 5 days in the 500-day window (these set the 99% tail):")
for d, r in ret.sort_values().head(5).items():
    print(f"  {d.date()}   ${r*POSITION:>10,.0f}   ({r*100:6.2f}%)")


## 3. Monte Carlo VaR
Simulate one-day AAPL returns from a model fitted to the sample, revalue the position,
and take the quantile. We show a normal MC (which should track the parametric number)
and a fat-tailed Student-t MC (which pushes the tail out).


In [ ]:
np.random.seed(42)
N_SIM = 100_000

# (a) Normal Monte Carlo - same assumption as parametric, so it should agree
sim_ret = np.random.normal(mu, sig, N_SIM)
sim_pnl = sim_ret * POSITION

def mc_var(alpha):
    return -np.percentile(sim_pnl, (1-alpha)*100)

for a in [0.95, 0.99]:
    v = mc_var(a)
    print(f"{a*100:.0f}% 1-day Monte Carlo VaR (Normal): ${v:>9,.0f}  ({v/POSITION*100:.2f}%)")

var_mc = mc_var(ALPHA)
print(f"Simulations: {N_SIM:,} (seed=42). Normal MC should match the parametric VaR.")

# (b) Student-t Monte Carlo - fatter tails, scaled to the same sigma
nu = 5
t_scale   = sig / np.sqrt(nu/(nu-2))                  # so simulated sd equals sample sigma
sim_ret_t = mu + t_scale * np.random.standard_t(nu, N_SIM)
var_mc_t  = -np.percentile(sim_ret_t*POSITION, (1-ALPHA)*100)
print(f"\n99% MC VaR under Student-t(nu=5): ${var_mc_t:,.0f}  vs ${var_mc:,.0f} normal")
print("Same mean and sd, but heavier tails give a larger 99% VaR - the fat-tail lesson.")


## 4. Visual Comparison of the Three Methods


In [ ]:
methods = ['Parametric', 'Historical', 'Monte Carlo']
vals99  = [var_param, var_hist, var_mc]
colors  = ['#4C78A8', '#F58518', '#54A24B']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (a) bar chart of the three 99% VaRs
axes[0].bar(methods, vals99, color=colors, alpha=0.85, edgecolor='white')
for i, v in enumerate(vals99):
    axes[0].text(i, v, f'${v:,.0f}', ha='center', va='bottom', fontweight='bold')
axes[0].set_title(f'{TICKER} - 1-day 99% VaR by Method (${POSITION:,.0f} position)')
axes[0].set_ylabel('VaR ($)')

# (b) P&L histogram with the three cut-lines
axes[1].hist(pnl, bins=50, color='#B0B0B0', alpha=0.7, edgecolor='white')
for m, v, c in zip(methods, vals99, colors):
    axes[1].axvline(-v, color=c, lw=2, ls='--', label=f'{m}: ${v:,.0f}')
axes[1].set_title(f'{TICKER} Daily P&L (last {WINDOW} days) with 99% VaR')
axes[1].set_xlabel('Daily P&L ($)'); axes[1].set_ylabel('Frequency')
axes[1].legend(frameon=False, fontsize=9)
plt.tight_layout(); plt.show()

print("Typical pattern: Parametric ~ Normal MC; Historical differs where AAPL's tails")
print("are fatter or skewed relative to the normal.")


## 5. Expected Shortfall (ES / CVaR)
ES is the average loss *beyond* VaR - it sees the size of the tail, not just its edge:
$$\text{ES}_\alpha = E[\,L \mid L > \text{VaR}_\alpha\,] \;\ge\; \text{VaR}_\alpha$$


In [ ]:
# Historical ES: average of losses beyond the VaR cut
def historical_es(alpha):
    q = np.percentile(pnl, (1-alpha)*100)
    return -pnl[pnl <= q].mean()

# Parametric (normal) ES: sigma * phi(z)/(1-alpha) - mu
def parametric_es(alpha):
    z = norm.ppf(alpha)
    return -(mu - sig*norm.pdf(z)/(1-alpha)) * POSITION

print(f"{'level':>6} | {'Hist VaR':>10} {'Hist ES':>10} | {'Param VaR':>10} {'Param ES':>10}")
print("-"*58)
for a in [0.95, 0.99]:
    print(f"{a*100:>5.0f}% | ${historical_var(a):>9,.0f} ${historical_es(a):>9,.0f} | "
          f"${parametric_var(a):>9,.0f} ${parametric_es(a):>9,.0f}")

print("\nES > VaR always: it averages the whole tail. The Historical ES is the honest")
print("read on AAPL's realised crash days; parametric ES trusts the normal tail.")


## 6. VaR Backtesting - Exceptions & the Basel Traffic Light
Does the model actually cover 99% of days? We run a **rolling** backtest: each day, estimate
99% Historical VaR from the trailing 250 days, then check whether the next day's loss breaches
it. Testing over the most recent 250 days uses the last 500 days in total.


In [ ]:
EST, TEST = 250, 250
alpha_bt  = 0.99
r = ret_all.iloc[-(EST+TEST):]                    # 250 to estimate + 250 to test

exceptions, breach_dates = 0, []
for t in range(EST, len(r)):
    window_pnl = r.iloc[t-EST:t] * POSITION
    var_t      = -np.percentile(window_pnl, (1-alpha_bt)*100)   # yesterday's VaR estimate
    loss_t     = -r.iloc[t] * POSITION                          # today's actual loss
    if loss_t > var_t:
        exceptions += 1
        breach_dates.append(r.index[t].date())

print(f"Rolling 99% VaR backtest over the last {TEST} days (250-day estimation window)")
print(f"Expected exceptions: {TEST}x1% = {TEST*(1-alpha_bt):.1f}    Observed: {exceptions}")

# Basel traffic light (250-day frame)
if   exceptions <= 4:  zone, mult = 'GREEN',  3.00
elif exceptions <= 9:  zone, mult = 'YELLOW', 3.00 + [0.40,0.50,0.65,0.75,0.85][exceptions-5]
else:                  zone, mult = 'RED',    4.00
print(f"Basel zone: {zone}    capital multiplier m_c = {mult:.2f}")
if breach_dates:
    print("Breach days:", ", ".join(str(d) for d in breach_dates))


## 7. Multi-Horizon VaR (Square-Root-of-Time Rule)
Scale the 1-day VaR to longer horizons under the i.i.d. assumption:
$$\text{VaR}_h = \text{VaR}_1\,\sqrt{h}$$


In [ ]:
horizons = [1, 5, 10, 21, 63, 252]
print(f"{TICKER} 99% VaR scaled from the 1-day parametric VaR (${var_param:,.0f}):\n")
print(f"{'Horizon (days)':>15} | {'VaR ($)':>14} | {'% of position':>14}")
print("-"*50)
for h in horizons:
    vh = var_param * np.sqrt(h)
    print(f"{h:>15} | {vh:>14,.0f} | {vh/POSITION*100:>13.2f}%")
print("\nsqrt-time assumes i.i.d., zero-drift returns: fine at short horizons, weaker as h grows")
print("(AAPL shows some volatility clustering, so a 1-year scale-up is only a rough guide).")


## Summary

| Method | Assumption | Reads the tail from | Best when |
|---|---|---|---|
| Parametric | Normal returns | mu and sigma only | tails are near-normal, need speed |
| Historical | None (empirical) | the actual 500 days | history contains the risks you fear |
| Monte Carlo | A fitted model | simulated draws | payoff is nonlinear or you want many scenarios |

**For AAPL over the last 500 days:** parametric and normal Monte Carlo agree by construction;
historical simulation and the Student-t Monte Carlo capture the fat tails the normal misses,
which is why they typically give the larger 99% loss. **Expected Shortfall** sits above every
VaR figure, and the **rolling backtest** tells you whether the model actually held up out of
sample. Always state the confidence level, the horizon, and the estimation window on every number.
